## 任务二

复用 `task1` 中的图像，计算以下间距：

a) 相邻两片电池（左右并排）之间的距离  
b) 相邻两片电池（上下叠放）之间的距离  
c) 在相机 1 和相机 4 中，最外侧电池到外缘汇流条的距离  
d) 在相机 2 和相机 3 中，内侧电池到中间汇流条的距离 

每个间隙需给出两个数值：该间隙的最小距离与最大距离。需将像素距离换算为毫米（mm），换算时假定每片电池的高度为 210 mm。


### 根据每片电池的高度计算每个像素占多少高度

取 **`task1/set2` 排序后第一张图**（`_0.jpg`）：**上边界为汇流条底边缘**（左侧条带行均值在亮带峰值之后相对电池片区的大幅下落处），**下边界为第一行与第二行之间的行缝**（行方向梯度投影的第二条细横线峰），量出第一行高度 **h**（像素），再按 **210 mm / h** 得到 **mm/px**。下图风格与您手写标注一致（左侧红双箭头 + **h**）。



In [ ]:
from task2 import CELL_HEIGHT_MM, first_cell_row_height_px
from task2_notebook_helpers import load_bgr, task1_set_paths

import cv2
import matplotlib.pyplot as plt

plt.rcParams["font.sans-serif"] = [
    "PingFang SC",
    "Heiti TC",
    "Songti SC",
    "STHeiti",
    "SimHei",
    "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False

import numpy as np

# set2 排序后第一张（_0）
fp = task1_set_paths("set2")[0]
bgr = load_bgr(fp)
gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
y0, y1, h_px = first_cell_row_height_px(gray)
mm_px = CELL_HEIGHT_MM / h_px

h_img, w = gray.shape
x_c = int(w * 0.10)
half = max(8, w // 120)
x_lo, x_hi = max(0, x_c - half), min(w, x_c + half + 1)
# 竖向剖面：与箭头同列附近的行平均灰度（能量）
profile = gray[:, x_lo:x_hi].mean(axis=1).astype(np.float64)
y_px = np.arange(h_img)

fig, (ax_img, ax_prof) = plt.subplots(
    1,
    2,
    figsize=(16, 7.2),
    constrained_layout=True,
)
fig.suptitle("第一行电池高度标定", fontsize=15, fontweight="bold")

ax_img.imshow(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
ax_img.annotate(
    "",
    xy=(x_c, y1),
    xytext=(x_c, y0),
    arrowprops=dict(arrowstyle="<->", color="red", lw=2.5, mutation_scale=16),
)
ax_img.text(
    x_c - w * 0.02,
    (y0 + y1) / 2,
    "h",
    fontsize=22,
    color="red",
    fontweight="bold",
    ha="right",
    va="center",
)
ax_img.set_title("示意图：汇流条底缘 → 第一行行缝", fontsize=13)
ax_img.axis("off")

ax_prof.plot(y_px, profile, color="steelblue", linewidth=0.9)
ax_prof.axvline(y0, color="red", linestyle="--", linewidth=1.2, alpha=0.9, label="汇流条底缘 y0")
ax_prof.axvline(y1, color="forestgreen", linestyle="--", linewidth=1.2, alpha=0.9, label="第一行行缝 y1")
ax_prof.set_xlabel("竖向像素位置 / px")
ax_prof.set_ylabel("能量（行平均灰度）")
ax_prof.set_title("竖向跨汇流条区域灰度变化", fontsize=13)
ax_prof.legend(loc="upper right", fontsize=9)
ax_prof.grid(True, alpha=0.35)

plt.show()

print(f"标定用图: {fp.name}")
print(f"第一行像素高度 h = {h_px} px（上缘 y0={y0}，行缝 y1={y1}）")
print(f"距离比例: 1 px ≈ {mm_px:.6f} mm（{CELL_HEIGHT_MM} mm / h）")


### a) 左右并排间隙


### a) / b) 检测与统计（与 `task2-set1.ipynb` 同一套代码）

**仅数据目录不同**：本 notebook 使用 **`task1/set2`**（`task1_set_paths("set2")`、`first_row_calibrated_mm_per_px("set2")`）。**a)、b) 的算法与 set1 完全一致**，均来自 `task2_notebook_helpers` → `task2_gap_measurement`。

- **a) 细竖线**：`thin_vertical_lines_gap_gradient` — `compute_lines` 得 x（剔除宽竖带）+ `measure_vline_gap_gradient`（v3 列均值带符号梯度正/负峰间距）。
- **b) 细横线**：`thin_horizontal_lines_gap_gradient` — 行能量 + 剔除粗横带；`measure_hline_gap_gradient`（v3）；与纵剖面一致的 `gy_smooth` 对齐；**中行**槽位 4–7 用 `gy_smooth` 强峰 NMS 补第三缝、**窄窗 hone**（跨距限制在细缝范围，抑制粗边假峰）；槽位决定每图 2 或 3 条。

下方 **c、d** 仍为列/行均值双阈值内缘，与 a/b 的梯度跨距法不同。


上方输出：**3×4 拼图**（绿线=检测 x）；**紧随其后的图**：每张一条横剖面，与检测同源 |gx|（左右汇流条列已压掉），**红虚线**为检测 x，便于对照 `_2`/`_10` 等是否误吃边带。

a) 与 `task2-set1.ipynb` **同一套** `thin_vertical_lines_gap_gradient`；原理见上一格「a) / b) 检测与统计」。


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display

from task2_notebook_helpers import (
    first_row_calibrated_mm_per_px,
    load_bgr,
    minmax_px_mm_with_source,
    mosaic_3x4,
    paint_vertical_lines,
    thin_vertical_lines_gap_gradient,
    vertical_seam_horizontal_profiles,
    task1_set_paths,
)

mm_per_px = first_row_calibrated_mm_per_px("set2")
rows_a: list[tuple[str, float]] = []
painted = []
paths = task1_set_paths("set2")
profile_xs: list[list[int]] = []
for p in paths:
    bgr = load_bgr(p)
    xs, spans = thin_vertical_lines_gap_gradient(bgr)
    profile_xs.append(xs)
    print(
        f"{p.name}: 细竖线 {len(xs)} 条, "
        f"梯度法跨像素 {spans}"
    )
    for si, s in enumerate(spans):
        rows_a.append((f"{p.name}#竖{si}", float(s)))
    painted.append(paint_vertical_lines(bgr, xs, thickness=2))

grid = mosaic_3x4(painted)
_, buf = cv2.imencode(".jpg", grid, [int(cv2.IMWRITE_JPEG_QUALITY), 88])
display(Image(data=buf.tobytes()))

fig, axes = plt.subplots(3, 4, figsize=(16, 9), sharex=True)
axes_flat = axes.ravel()
for ax, p, xs in zip(axes_flat, paths, profile_xs):
    bgr = load_bgr(p)
    g = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    prof = vertical_seam_horizontal_profiles(g)
    x = prof["x"]
    gx = prof["gx_smooth"]
    gray = prof["gray_mean"]
    inv = prof["inv_mean"]
    ax.plot(x, gx / (np.max(gx) + 1e-9), label="|gx| smooth (norm)", color="C0", linewidth=1)
    ax.plot(x, gray / 255.0, label="灰度/255", color="C1", alpha=0.7, linewidth=0.8)
    ax.plot(x, inv / 255.0, label="(255-I)/255", color="C2", alpha=0.7, linewidth=0.8)
    for xv in xs:
        ax.axvline(xv, color="red", linestyle="--", linewidth=0.8, alpha=0.85)
    ax.set_title(p.name, fontsize=8)
    ax.set_ylim(-0.05, 1.05)
axes_flat[0].legend(loc="upper right", fontsize=6)
for ax in axes[-1]:
    ax.set_xlabel("x (px)")

plt.suptitle("set2 · 横剖面与细竖线 x（定位：task2_gap_measurement / 统计：梯度跨像素）", fontsize=11)
fig.tight_layout(rect=[0, 0.02, 1, 0.93])
plt.show()

minmax_px_mm_with_source(
    rows_a, mm_per_px, "a) set2·左右并排·细竖缝梯度跨距 · 极值"
)


### b) 上下叠放间隙


## 读取 `task1/set*` 共 12 张图

**细横线（task2_gap_measurement）**：行能量上先 **剔除一道粗横条带**（与竖向剔除宽竖条对称，抑制边缘粗横线），对检出 y 使用 ``measure_hline_gap_gradient``（v3）及剖面 ``gy_smooth`` 对齐、**窄窗 hone**（细缝跨距约束）；**中行** 槽位 4–7 另用 ``gy_smooth`` 强峰补第三缝（见上格「a) / b) 检测与统计」）。按槽位 **上行/下行各 2 条、中行 3 条**（非中行在细跨距候选中取 **y 较大** 的 2 条）。

**本格应看到两种输出**：① **JPEG 拼图**（12 张缩略）；② **3×4 纵剖面**（12 张子图）。若第二张大图被裁切只像 11 格，已用 `fig.tight_layout(rect=...)` 为 suptitle 留边；仍异常请重跑。

**与 set1**：同一 `thin_horizontal_lines_gap_gradient`；本文件为 **`task1/set2`**。


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display

from task2_notebook_helpers import (
    first_row_calibrated_mm_per_px,
    horizontal_seam_vertical_profiles,
    load_bgr,
    minmax_px_mm_with_source,
    mosaic_3x4,
    paint_horizontal_lines,
    task1_set_paths,
    thin_horizontal_lines_gap_gradient,
    _slot_key,
)

mm_per_px = first_row_calibrated_mm_per_px("set2")
rows_b: list[tuple[str, float]] = []
painted = []
paths = task1_set_paths("set2")
profile_ys: list[list[int]] = []
for p in paths:
    bgr = load_bgr(p)
    ys, spans = thin_horizontal_lines_gap_gradient(bgr, _slot_key(p))
    profile_ys.append(ys)
    print(
        f"{p.name}: 细横线 {len(ys)} 条, "
        f"梯度法跨像素 {spans}"
    )
    for si, s in enumerate(spans):
        rows_b.append((f"{p.name}#横{si}", float(s)))
    painted.append(paint_horizontal_lines(bgr, ys, thickness=2))

grid = mosaic_3x4(painted)
_, buf = cv2.imencode(".jpg", grid, [int(cv2.IMWRITE_JPEG_QUALITY), 88])
display(Image(data=buf.tobytes()))

fig, axes = plt.subplots(3, 4, figsize=(16, 9), sharex=True)
axes_flat = axes.ravel()
for ax, p, ys in zip(axes_flat, paths, profile_ys):
    bgr = load_bgr(p)
    g = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    prof = horizontal_seam_vertical_profiles(g)
    y = prof["y"]
    gy = prof["gy_smooth"]
    gray = prof["gray_mean"]
    inv = prof["inv_mean"]
    ax.plot(y, gy / (np.max(gy) + 1e-9), label="|gy| smooth (norm)", color="C0", linewidth=1)
    ax.plot(y, gray / 255.0, label="灰度/255", color="C1", alpha=0.7, linewidth=0.8)
    ax.plot(y, inv / 255.0, label="(255-I)/255", color="C2", alpha=0.7, linewidth=0.8)
    for yv in ys:
        ax.axvline(yv, color="red", linestyle="--", linewidth=0.8, alpha=0.85)
    ax.set_title(p.name, fontsize=8)
    ax.set_ylim(-0.05, 1.05)
axes_flat[0].legend(loc="upper right", fontsize=6)
for ax in axes[-1]:
    ax.set_xlabel("y (px)")

plt.suptitle("set2 · 纵剖面与细横线 y（定位：task2_gap_measurement / 统计：梯度跨像素）", fontsize=11)
fig.tight_layout(rect=[0, 0.02, 1, 0.93])
plt.show()

minmax_px_mm_with_source(
    rows_b, mm_per_px, "b) set2·上下叠放·细横缝梯度跨距 · 极值"
)


### c) 相机 1、4：外缘竖直汇流条宽度


## 相机 1、4 共 6 张图

与 **d)** 相同：**列均值剖面**（先平滑），在左/右 ROI 内定竖向外缘汇流条；**两内缘之间**为宽度。另按槽位在整板 3×4 中的 **行**：**上行 tile（0–3）** 增加 **上缘横向** 行均值剖面，**下行 tile（8–11）** 增加 **下缘横向**，**中行（4–7）** 仅保留左右竖向（与 `_4`、`_7` 一致）。下方每张图先输出竖向，再按需输出横向。


In [ ]:
import io
import math
import re

import cv2
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image, display

plt.rcParams["font.sans-serif"] = [
    "PingFang SC",
    "Heiti TC",
    "Songti SC",
    "STHeiti",
    "SimHei",
    "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False

from task2_notebook_helpers import (
    CAM1_LEFT_BROAD_PEAK_SLOTS,
    minmax_px_mm_with_source,
    busbar_edges_from_column_mean,
    busbar_edges_from_row_mean,
    busbar_search_roi_cam14,
    busbar_search_roi_row_top_bottom,
    cam14_horizontal_edge_kinds,
    cam_paths,
    load_bgr,
    first_row_calibrated_mm_per_px,
)


def suffix_n(p):
    return int(re.search(r"_(\d+)\.", p.name).group(1))


paths = cam_paths(1, "set2") + cam_paths(4, "set2")
if not paths:
    raise FileNotFoundError("cam_paths(1)+cam_paths(4) 为空，请确认 task1/set2 下有图")

mm_per_px = first_row_calibrated_mm_per_px("set2")
rows_c: list[tuple[str, float]] = []

for p in paths:
    bgr = load_bgr(p)
    g = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    h, w = g.shape
    mode = "left" if suffix_n(p) % 4 == 0 else "right"
    tag = "相机1 左缘" if mode == "left" else "相机4 右缘"
    x0, x1 = busbar_search_roi_cam14(w, mode)
    sn = suffix_n(p)
    min_run = (
        32
        if mode == "left" and sn in CAM1_LEFT_BROAD_PEAK_SLOTS
        else None
    )
    prof, e = busbar_edges_from_column_mean(
        g, x0, x1, min_peak_run_px=min_run
    )
    if not math.isnan(e.thr_big):
        rows_c.append((f"{p.name}:外缘竖向", float(e.width_inner_px)))
    x_px = np.arange(w, dtype=float)

    fig, (ax_img, ax_prof) = plt.subplots(
        1, 2, figsize=(16, 6.8), constrained_layout=True
    )
    fig.suptitle(
        f"c) 外缘汇流条 — {tag} 竖向 — {p.name}\n"
        f"汇流条宽（内缘间）= {e.width_inner_px} px",
        fontsize=13,
        fontweight="bold",
    )

    ax_img.imshow(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
    ax_img.axvline(e.x_left_big, color="crimson", linestyle="--", linewidth=1.1, alpha=0.95)
    ax_img.axvline(e.x_left_small, color="darkorange", linestyle="--", linewidth=1.1, alpha=0.95)
    ax_img.axvline(e.x_right_small, color="darkorange", linestyle="--", linewidth=1.1, alpha=0.95)
    ax_img.axvline(e.x_right_big, color="crimson", linestyle="--", linewidth=1.1, alpha=0.95)
    ax_img.axvspan(e.x_left_small, e.x_right_small, alpha=0.22, color="gold")
    ax_img.set_title("原图（金区=竖向汇流条）", fontsize=12)
    ax_img.axis("off")

    ax_prof.plot(x_px, prof, color="steelblue", linewidth=0.85, label="列均值（平滑）")
    ax_prof.axvline(e.x_left_big, color="crimson", linestyle="--", linewidth=1.0, alpha=0.9, label="突变大(外缘)")
    ax_prof.axvline(e.x_left_small, color="darkorange", linestyle="--", linewidth=1.0, alpha=0.9, label="突变小(内缘)")
    ax_prof.axvline(e.x_right_small, color="darkorange", linestyle=":", linewidth=1.0, alpha=0.9)
    ax_prof.axvline(e.x_right_big, color="crimson", linestyle="--", linewidth=1.0, alpha=0.9)
    ax_prof.axvspan(e.x_left_small, e.x_right_small, alpha=0.18, color="gold")
    ax_prof.axvline(e.x_peak, color="lime", linestyle="-", linewidth=1.0, alpha=0.85, label=f"峰 x={e.x_peak}")
    ax_prof.set_xlabel("横向像素位置 / px")
    ax_prof.set_ylabel("灰度（列平均）")
    ax_prof.set_title("列均值剖面：大/小突变点与汇流条区", fontsize=12)
    ax_prof.legend(loc="upper right", fontsize=8)
    ax_prof.grid(True, alpha=0.35)

    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=120, bbox_inches="tight")
    plt.close(fig)
    display(Image(data=buf.getvalue()))

    for hk in cam14_horizontal_edge_kinds(sn):
        try:
            y0, y1 = busbar_search_roi_row_top_bottom(h, hk)
            min_run_h = (
                32
                if mode == "left" and sn in CAM1_LEFT_BROAD_PEAK_SLOTS
                else None
            )
            prof_r, er = busbar_edges_from_row_mean(
                g, y0, y1, min_peak_run_px=min_run_h
            )
            if math.isnan(er.thr_big):
                print(f"  [跳过] {p.name} {hk}: 行向剖面未得到有效双阈值")
                continue
            htag = "上缘横向" if hk == "top" else "下缘横向"
            rows_c.append((f"{p.name}:{htag}", float(er.width_inner_px)))
            y_px = np.arange(h, dtype=float)

            fig2, (ax_img2, ax_prof2) = plt.subplots(
                1, 2, figsize=(16, 6.8), constrained_layout=True
            )
            fig2.suptitle(
                f"c) 外缘汇流条 — {tag} {htag} — {p.name}\n"
                f"汇流条宽（内缘间，行向）= {er.width_inner_px} px",
                fontsize=13,
                fontweight="bold",
            )
            ax_img2.imshow(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
            ax_img2.axhline(er.x_left_big, color="crimson", linestyle="--", linewidth=1.1, alpha=0.95)
            ax_img2.axhline(er.x_left_small, color="darkorange", linestyle="--", linewidth=1.1, alpha=0.95)
            ax_img2.axhline(er.x_right_small, color="darkorange", linestyle="--", linewidth=1.1, alpha=0.95)
            ax_img2.axhline(er.x_right_big, color="crimson", linestyle="--", linewidth=1.1, alpha=0.95)
            ax_img2.axhspan(er.x_left_small, er.x_right_small, alpha=0.22, color="gold")
            ax_img2.set_title("原图（金区=横向汇流条）", fontsize=12)
            ax_img2.axis("off")

            ax_prof2.plot(y_px, prof_r, color="steelblue", linewidth=0.85, label="行均值（平滑）")
            ax_prof2.axvline(er.x_left_big, color="crimson", linestyle="--", linewidth=1.0, alpha=0.9, label="突变大(外缘)")
            ax_prof2.axvline(er.x_left_small, color="darkorange", linestyle="--", linewidth=1.0, alpha=0.9, label="突变小(内缘)")
            ax_prof2.axvline(er.x_right_small, color="darkorange", linestyle=":", linewidth=1.0, alpha=0.9)
            ax_prof2.axvline(er.x_right_big, color="crimson", linestyle="--", linewidth=1.0, alpha=0.9)
            ax_prof2.axvspan(er.x_left_small, er.x_right_small, alpha=0.18, color="gold")
            ax_prof2.axvline(er.x_peak, color="lime", linestyle="-", linewidth=1.0, alpha=0.85, label=f"峰 y={er.x_peak}")
            ax_prof2.set_xlabel("竖向像素位置 y / px")
            ax_prof2.set_ylabel("灰度（行平均）")
            ax_prof2.set_title("行均值剖面：大/小突变点与汇流条区", fontsize=12)
            ax_prof2.legend(loc="upper right", fontsize=8)
            ax_prof2.grid(True, alpha=0.35)

            buf2 = io.BytesIO()
            fig2.savefig(buf2, format="png", dpi=120, bbox_inches="tight")
            plt.close(fig2)
            display(Image(data=buf2.getvalue()))
        except Exception as ex:
            print(f"  [错误] {p.name} {hk} 横向: {type(ex).__name__}: {ex}")

minmax_px_mm_with_source(
    rows_c,
    mm_per_px,
    "c) set2·相机1/4·外缘汇流条（内缘，含上下横补）·极值",
)


### d) 相机 2、3：中间竖直汇流条宽度


## 相机 2、3 共 6 张图

与 **c)** 相同：**列均值剖面 + 突变大/小点**；搜索区间为**整幅宽度**。下方输出 6 张大图。




In [ ]:
import io
import re

import cv2
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image, display

plt.rcParams["font.sans-serif"] = [
    "PingFang SC",
    "Heiti TC",
    "Songti SC",
    "STHeiti",
    "SimHei",
    "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False

from task2_notebook_helpers import (
    busbar_edges_from_column_mean,
    cam_paths,
    minmax_px_mm_with_source,
    load_bgr,
    first_row_calibrated_mm_per_px,
)


def suffix_n(p):
    return int(re.search(r"_(\d+)\.", p.name).group(1))


paths = cam_paths(2, "set2") + cam_paths(3, "set2")
if not paths:
    raise FileNotFoundError("cam_paths(2)+cam_paths(3) 为空，请确认 task1/set2 下有图")

mm_per_px = first_row_calibrated_mm_per_px("set2")
rows_d: list[tuple[str, float]] = []

for p in paths:
    bgr = load_bgr(p)
    g = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    h, w = g.shape
    n = suffix_n(p)
    tag = "相机2" if n % 4 == 1 else "相机3"
    prof, e = busbar_edges_from_column_mean(g, 0, w)
    rows_d.append((p.name, float(e.width_inner_px)))
    x_px = np.arange(w, dtype=float)

    fig, (ax_img, ax_prof) = plt.subplots(
        1, 2, figsize=(16, 6.8), constrained_layout=True
    )
    fig.suptitle(
        f"d) 中间汇流条 — {tag} — {p.name}\n"
        f"汇流条宽（内缘间）= {e.width_inner_px} px",
        fontsize=13,
        fontweight="bold",
    )

    ax_img.imshow(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
    ax_img.axvline(e.x_left_big, color="crimson", linestyle="--", linewidth=1.1, alpha=0.95)
    ax_img.axvline(e.x_left_small, color="darkorange", linestyle="--", linewidth=1.1, alpha=0.95)
    ax_img.axvline(e.x_right_small, color="darkorange", linestyle="--", linewidth=1.1, alpha=0.95)
    ax_img.axvline(e.x_right_big, color="crimson", linestyle="--", linewidth=1.1, alpha=0.95)
    ax_img.axvspan(e.x_left_small, e.x_right_small, alpha=0.22, color="gold")
    ax_img.set_title("原图（金区=汇流条区域）", fontsize=12)
    ax_img.axis("off")

    ax_prof.plot(x_px, prof, color="steelblue", linewidth=0.85, label="列均值（平滑）")
    ax_prof.axvline(e.x_left_big, color="crimson", linestyle="--", linewidth=1.0, alpha=0.9, label="突变大(外缘)")
    ax_prof.axvline(e.x_left_small, color="darkorange", linestyle="--", linewidth=1.0, alpha=0.9, label="突变小(内缘)")
    ax_prof.axvline(e.x_right_small, color="darkorange", linestyle=":", linewidth=1.0, alpha=0.9)
    ax_prof.axvline(e.x_right_big, color="crimson", linestyle="--", linewidth=1.0, alpha=0.9)
    ax_prof.axvspan(e.x_left_small, e.x_right_small, alpha=0.18, color="gold")
    ax_prof.axvline(e.x_peak, color="lime", linestyle="-", linewidth=1.0, alpha=0.85, label=f"峰 x={e.x_peak}")
    ax_prof.set_xlabel("横向像素位置 / px")
    ax_prof.set_ylabel("能量（列平均灰度）")
    ax_prof.set_title("列均值剖面：大/小突变点与汇流条区", fontsize=12)
    ax_prof.legend(loc="upper right", fontsize=8)
    ax_prof.grid(True, alpha=0.35)

    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=120, bbox_inches="tight")
    plt.close(fig)
    display(Image(data=buf.getvalue()))

minmax_px_mm_with_source(
    rows_d,
    mm_per_px,
    "d) set2·相机2/3·中间汇流条（内缘）·极值",
)



### set2 · 四项极值一览

与上方 a–d 分步相同数据；若已跑完 a–d，此处结果应一致。


In [ ]:
from task2_notebook_helpers import print_abcd_minmax_summary_for_set

print_abcd_minmax_summary_for_set("set2")


### 与其他 notebook 对照

本文件处理 **`task1/set2`**。**a、b 与 `task2-set1.ipynb` 使用相同计算方法**（同一 helper，仅换图集与标定）。c、d 逻辑亦一致。数值可与其它 `task2-set*.ipynb` 对照。


In [ ]:
# 本 notebook 专注 set2；其它集合请打开对应 task2-set*.ipynb
pass
